# DSO 576 Module 6 — Amazon Beauty Product Reviews
**Student:** Lame Wadipeba

The cleaning rules are stored in `cleaning_script.py`. This notebook loads the original data, calls the cleaning function from the `.py` file, and documents the verification results.

## 1. Project vision and question

**Project vision:** Understand customer satisfaction with beauty products by analyzing Amazon review ratings and review characteristics to identify patterns in positive and negative customer experiences.

**Business question:** How do review characteristics, such as verified purchase status and helpful votes, relate to customer ratings for beauty products?

**Main metrics:** Average rating, rating distribution, verified vs. unverified purchase ratings, helpful votes, and review length.

## 2. Load the original source

The source is a 4,687-row sample of Amazon Beauty reviews. The original dataframe is kept unchanged in `df_original`; the cleaning function works on a copy.

In [ ]:
import pandas as pd
from cleaning_script import load_source_data, clean_data, validate_data, make_record_checks, save_outputs

df_original = load_source_data()
print("Rows:", len(df_original))
print("Columns:", len(df_original.columns))

## 3. Initial inspection

I inspected the columns, data types, missing values, duplicates, ranges, and text formatting before applying the cleaning rules.

In [ ]:
df_original.info()

print("\nMissing values by column:")
print(df_original.isna().sum())

print("\nExact duplicates:", df_original.duplicated().sum())
print("Rating values:", sorted(df_original["rating"].unique()))
print("Verified purchase counts:")
print(df_original["verified_purchase"].value_counts())
print("Helpful vote range:", df_original["helpful_vote"].min(), "to", df_original["helpful_vote"].max())
print("Negative helpful votes:", (df_original["helpful_vote"] < 0).sum())
print("Negative prices:", (df_original["product_price"] < 0).sum())
print("HTML <br> reviews:", df_original["review_text"].str.contains("<br", regex=False, na=False).sum())
converted = pd.to_datetime(df_original["timestamp"], unit="ms", errors="coerce")
print("Timestamp conversion failures:", converted.isna().sum())
print("Earliest timestamp:", converted.min())
print("Latest timestamp:", converted.max())

## 4. Apply cleaning rules from `cleaning_script.py`

The notebook does not repeat the cleaning code. It calls `clean_data()`, so the `.py` file is the source of truth for the cleaning rules.

In [ ]:
df = clean_data(df_original)
print("Starting rows:", len(df_original))
print("Final rows:", len(df))
print("Rows removed:", len(df_original) - len(df))
print("Columns after cleaning:", len(df.columns))
df.head()

## 5. Evidence & verification

These checks are based on the original data and the cleaned data. They are included so the cleaning decisions in the PDF can be checked directly from the notebook.

In [ ]:
validation = validate_data(df_original, df)
print("=== EVIDENCE & VERIFICATION ===")

print("\n1. DATASET SIZE")
print("Original rows:", validation["starting_rows"])
print("Original columns:", validation["starting_columns"])
print("Final rows:", validation["final_rows"])

print("\n2. MISSING VALUES IN ORIGINAL DATA")
print(df_original.isna().sum())

print("\n3. EXACT DUPLICATES")
print("Exact duplicate rows:", validation["exact_duplicates"])

print("\n4. DATA TYPES")
print(df_original.dtypes)

print("\n5. RATING CHECK")
print("Rating values:", sorted(df_original["rating"].unique()))
print("Invalid ratings:", validation["invalid_ratings"])

print("\n6. VERIFIED PURCHASE")
print("Original counts:")
print(df_original["verified_purchase"].value_counts())
print("Cleaned counts:")
print(df["verified_purchase"].value_counts())

print("\n7. HELPFUL VOTES")
print("Minimum:", df_original["helpful_vote"].min())
print("Maximum:", df_original["helpful_vote"].max())
print("Negative values:", validation["negative_helpful_votes"])

print("\n8. TIMESTAMP CHECK")
converted = pd.to_datetime(df_original["timestamp"], unit="ms", errors="coerce")
print("Failed conversions:", validation["timestamp_conversion_failures"])
print("Earliest:", converted.min())
print("Latest:", converted.max())

print("\n9. PRICE CHECK")
print("Negative prices:", validation["negative_prices"])

print("\n10. TEXT CHECK")
print("Reviews with HTML line breaks in original:", validation["html_breaks_original"])
print("HTML line breaks remaining after cleaning:", df["review_text"].str.contains("<br", regex=False, na=False).sum())

print("\n11. ROW RECONCILIATION")
print("Rows removed:", len(df_original) - len(df))
print("Expected rows after removing duplicates:", len(df_original) - validation["exact_duplicates"])
print("Actual final rows:", len(df))

## 6. Metrics

These are the main metrics for the project question.

In [ ]:
print("Average rating:", df["rating"].mean())
print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())
print("\nVerified vs. unverified ratings:")
print(df.groupby("verified_purchase")["rating"].agg(["count", "mean"]))
print("\nAverage helpful votes:", df["helpful_vote"].mean())
print("Average review length:", df["review_length"].mean())

## 7. Five record checks

These checks compare an original record or cleaning decision with the expected result and the actual cleaned result.

In [ ]:
checks = make_record_checks(df_original, df)
checks

## 8. Save outputs

The same functions can be run from the command line through `cleaning_script.py`. The notebook saves the same cleaned files for easy inspection and submission.

In [ ]:
out = save_outputs(df_original, df)
print("Outputs saved to:", out.resolve())
print("\nOutput files:")
for file in sorted(out.iterdir()):
    print(file.name)